In [ ]:
import os
import re
from pathlib import Path
import wikipediaapi
import json
from typing import List


In [ ]:
USER_AGENT = os.getenv('Learning project: TourGuideAI/1.0', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
                          ' Chrome/91.0.4472.124 Safari/537.36')

In [ ]:
URL_PATH_WIKI = Path('../data/wiki')
def load_urls(path: Path):
    urls = []
    for file in path.glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

            for entry in data:
                urls.append(entry["url"])
    return urls
urls_wiki = load_urls(URL_PATH_WIKI)
urls_wiki

In [ ]:
wiki = wikipediaapi.Wikipedia(language='de',
                             extract_format=wikipediaapi.ExtractFormat.WIKI,
                             user_agent=USER_AGENT)

def load_wikipedia_text(title: str) -> dict:
    page = wiki.page(title)
    if not page.exists():
        return {'error': 'Page not found'}
    text = page.text
    return {
        'text': text,
        'title': page.title,
        'source': page.fullurl,
        'license': 'CC BY-SA 4.0'
    }

#Titel aus URL extrahieren
def extract_title_from_url(url: str) -> str:
    match = re.search(r'/wiki/([^#]+)', url)
    if match:
        return match.group(1)
    return None

def get_summary_from_wiki(text:str, n=2) -> str:
    paragraphs = text.split('\n\n')
    summary = ' '.join(paragraphs[:n])
    return summary

wiki_data = {}
for url in urls_wiki:
    title = extract_title_from_url(url)
    wiki_data[title] = load_wikipedia_text(title)


# Ergebnis als JSON ausgeben

#print(json.dumps(wiki_data, indent=2, ensure_ascii=False))
print(f"\n {len(wiki_data)} Wikipedia-Artikel geladen.")
for i, (title, data) in enumerate(wiki_data.items(),start=1):
    if 'text' in data:
        summary = get_summary_from_wiki(data['text'])
        print(f"\n--- Artikel {i}: ---")
        print(f"\n {data['title']}\n{summary}\n")

In [ ]:
def clean_text(text):
     # Kleinbuchstaben, Entfernen von Sonderzeichen
    text = text.lower().replace(",", " ").replace("!", " ").replace("?", " ").replace("-", " ").strip()

    # text in Liste von Wörtern umwandeln
    text = text.split()

    return text

for title, data in wiki_data.items():
        if 'text' in data:
            print(clean_text(data['text']))
        else:                     # Fehler abfangen, sonst Error
            print(f"Fehler bei {title}: {data.get('error', 'Unbekannter Fehler')}")

In [ ]:
def chunk_text(text: str, max_words: int = 200, overlap: int = 50) -> List[str]:
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + max_words
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap  # Überlappung für Kontext
        if start < 0:
            start = 0

    return chunks

chunks = []

for data in wiki_data.values():
    if 'text' in data:
        chunks.extend(chunk_text(data['text'], max_words=200, overlap=50))
print(f"{len(chunks)} Text-Chunks aus Wikipedia-Artikeln erstellt.")

In [ ]:
#Chunking mit metadaten
wiki_chunks = []
for title, data in wiki_data.items():
    if 'text' in data:
        chunks = chunk_text(data['text'], max_words=200, overlap=50)
        for i, chunk in enumerate(chunks):
            wiki_chunks.append({
                "text": chunk,
                "title": data['title'],
                "url": data['source']
            })
print(f"{len(wiki_chunks)} Text-Chunks mit Metadaten erstellt.")

In [ ]:
#Embeddings für Wiki-Chunks erstellen
from sentence_transformers import SentenceTransformer
model = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(model)
wiki_embeddings = embedding_model.encode([chunk['text'] for chunk in wiki_chunks])
print(f"Embeddings für {len(wiki_embeddings)} Wiki-Chunks erstellt.")
